# Notebook 02 — Run the four conditions
**Protocol:** `evaluation-design.md` (pre-locked). Conditions: **C1** full system (no memory, no critic) · **C2** same LLM, no retrieval · **C3** C1 without the re-ranker · **C4** C1 + critic–refinement pass.

You need: (1) your **Groq API key** (single key cell below — held in memory only, never written to disk or outputs); (2) the frozen `question_set.jsonl` from Notebook 01 (upload it when prompted).

The notebook refuses to run until the **Finding A category-filter correction** is applied and asserted in the cloned code.

In [ ]:
# --- Setup ---
!git clone -q https://github.com/vinurajd/AAIDC.git
%cd AAIDC
!pip install -q langchain langchain-community langchain-huggingface langchain-groq faiss-cpu sentence-transformers pypdf
import os, sys, json, getpass
os.environ['GROQ_API_KEY'] = getpass.getpass('Groq API key (input hidden, kept in memory only): ')
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
!git log -1 --format='commit %H %ci'

In [ ]:
# --- Upload the frozen question set ---
from google.colab import files
up = files.upload()   # choose question_set.jsonl
import hashlib
raw = open('question_set.jsonl', 'rb').read()
print('question_set.jsonl sha256 =', hashlib.sha256(raw).hexdigest())
ITEMS = [json.loads(l) for l in raw.decode().splitlines() if l.strip()]
print('Items loaded:', len(ITEMS))

In [ ]:
# --- Finding A correction: address categories BY NAME in tools.py (asserted) ---
import re
p = 'src/tools.py'
src = open(p).read()
before = src
# knowledge_tool already uses index 0 == 'Literature'; make all three name-based and unambiguous:
src = src.replace('filter={"category": list(categories_mapper.keys())[0]}', 'filter={"category": "Literature"}')
src = src.replace('filter={"category": list(categories_mapper.keys())[1]}', 'filter={"category": "Raga"}')
src = src.replace('filter={"category": list(categories_mapper.keys())[2]}', 'filter={"category": "Krithis"}')
assert src != before and 'keys())[1]' not in src and 'keys())[2]' not in src, 'Patch did not apply as expected'
open(p, 'w').write(src)
print('Finding A correction applied: raga_index_tool -> Raga, krithi_tool -> Krithis, knowledge_tool -> Literature')

In [ ]:
# --- Runner: mirrors src/app.py's flow exactly, with the two protocol toggles ---
import time, datetime
import tools as car_tools
from tools import knowledge_tool, krithi_tool, raga_index_tool, multi_search
from semantic_layer import Prompt, ReactAgent
from models import Models

models_obj = Models()
llm = models_obj.getLLM()
MODEL_STRING = 'llama3-8b-8192'   # from src/utils.py; recorded per the design document
print('LLM ready:', MODEL_STRING)

def select_tools(user_question):
    # Identical logic to src/app.py select_tools()
    q = user_question.lower(); sel = []
    if any(k in q for k in ["what is", "explain", "theory", "literature", "raga", "scale", "prayoga"]): sel.append(("knowledge_tool", knowledge_tool))
    if any(k in q for k in ["raga", "melakarta", "janya", "alias", "list", "number"]): sel.append(("raga_index_tool", raga_index_tool))
    if any(k in q for k in ["krithi", "kriti", "composition", "lyrics", "composer", "tala", "song"]): sel.append(("krithi_tool", krithi_tool))
    if len(sel) > 1 or "carnatic music" in q: sel.append(("multi_search", multi_search))
    if not sel: sel.append(("knowledge_tool", knowledge_tool))
    return sel

def run_retrieval(question):
    sel = select_tools(question); results = []; retrieved = []
    for name, fn in sel:
        if name == 'multi_search':
            r = multi_search.invoke({'query': question, 'categories': ['Literature', 'Raga', 'Krithis'], 'k_each': 4})
        else:
            r = fn.invoke(question)
        results.append(f'Results from {name}:\n{r}')
    return sel, results

def answer_with_retrieval(question, use_critic=False):
    sel, tool_results = run_retrieval(question)
    prompt_str = Prompt(question).getPromptStr()
    final_prompt = f"""{prompt_str}

Retrieved Information from Knowledge Base:
{chr(10).join(tool_results)}

Please provide a comprehensive answer based on the information above."""
    resp = llm.invoke(final_prompt).content
    if use_critic:
        ra = ReactAgent(llm)
        out = ra.process_with_react(question, resp, tool_results, 'No previous conversation.')
        resp = out['refined_response']
    return resp, [name for name, _ in sel], tool_results

def answer_no_retrieval(question):
    prompt = f'The following is a question about Carnatic music. Answer it directly.\n\nQuestion: {question}'
    return llm.invoke(prompt).content, [], []

In [ ]:
# --- Execute all four conditions over the frozen set ---
# C3 disables the re-ranker by patching the function the tools call (restored immediately after).
_orig_rerank = car_tools.re_rank_documents
def no_rerank(query, docs, top_k=6):
    return docs[:top_k]

os.makedirs('results', exist_ok=True)
log = open('results/run_log.jsonl', 'w')
for item in ITEMS:
    q = item['question']
    for cond in ['C1', 'C2', 'C3', 'C4']:
        if cond == 'C1':
            ans, tools_used, tres = answer_with_retrieval(q, use_critic=False)
        elif cond == 'C2':
            ans, tools_used, tres = answer_no_retrieval(q)
        elif cond == 'C3':
            car_tools.re_rank_documents = no_rerank
            try: ans, tools_used, tres = answer_with_retrieval(q, use_critic=False)
            finally: car_tools.re_rank_documents = _orig_rerank
        else:
            ans, tools_used, tres = answer_with_retrieval(q, use_critic=True)
        log.write(json.dumps({
            'item_id': item['item_id'], 'stratum': item['stratum'], 'condition': cond,
            'question': q, 'gold_answer': item['gold_answer'],
            'evidence_chunk_id': item.get('evidence_chunk_id', ''),
            'tools_selected': tools_used, 'tool_results': tres, 'answer': ans,
            'model': MODEL_STRING, 'decoding': 'service defaults (committed code sets neither temperature nor top_p)',
            'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
        }) + '\n'); log.flush()
    print('done', item['item_id'])
log.close()
print('Run complete -> results/run_log.jsonl')
from google.colab import files
files.download('results/run_log.jsonl')

**Share back** `run_log.jsonl` together with `question_set.jsonl`. Note for the record: tool results are logged verbatim (including the category/source labels the tools print), which is what makes routing accuracy and recall@6 computable in Notebook 03, and keeps any residual misrouting visible rather than hidden.